# 57. The Periodic Review (Base-Stock) Policy Problem

## Tier 3: The Advanced Algorithm (Genetic Algorithm for Multi-Objective Optimization)

### Key assumptions
- Multiple conflicting objectives need to be optimized simultaneously
- Trade-offs exist between cost, service level, inventory investment, and operational complexity
- Decision makers need a set of Pareto-optimal solutions to choose from
- Population-based search can explore complex solution spaces effectively

### Approach (step-by-step)
1. **Multi-Objective Formulation**: Define 4 conflicting objectives (cost, service, inventory, simplicity)
2. **NSGA-II Algorithm**: Implement Non-dominated Sorting Genetic Algorithm II
3. **Population Evolution**: Use genetic operators (crossover, mutation) for solution exploration
4. **Pareto Front Extraction**: Identify non-dominated solutions
5. **Decision Support**: Provide multiple solutions for different strategic preferences

### What to look for in the results
- Multiple Pareto-optimal solutions showing different trade-offs
- Clear visualization of the conflict between objectives
- Decision support for selecting appropriate policies
- Convergence analysis showing algorithm performance

### Concrete example (from the source)
Expected Pareto-optimal solutions:
- **High Service Focus**: R=0.85 weeks, S=26,843 units, Cost=$2,456/week, Service=97.8%
- **Balanced**: R=1.28 weeks, S=23,156 units, Cost=$2,289/week, Service=94.9%
- **Cost-Focused**: R=2.15 weeks, S=21,432 units, Cost=$2,199/week, Service=92.5%

### Why this Tier exists vs Tiers 1-2
The multi-objective GA addresses critical limitations of previous approaches:
- **Multiple Objectives**: Real-world decisions involve trade-offs beyond pure cost minimization
- **Strategic Flexibility**: Different strategies (service-focused vs cost-focused) can be evaluated
- **Pareto Optimality**: Provides mathematically optimal trade-off solutions
- **Decision Support**: Helps managers choose policies aligned with strategic priorities

### Pros / Cons vs Tiers 1-2
**Pros:**
- Handles multiple conflicting objectives simultaneously
- Provides a set of optimal solutions rather than single answer
- Explores complex solution spaces more thoroughly
- Supports strategic decision making with trade-off analysis
- Can handle non-linear relationships and constraints

**Cons:**
- Computationally intensive (population-based search)
- Requires parameter tuning (population size, generations, etc.)
- Results may vary between runs (stochastic algorithm)
- More complex to implement and understand
- May not guarantee convergence to global Pareto front

### When to use this Tier
- When multiple objectives are important (cost, service, inventory, complexity)
- When strategic trade-offs need to be evaluated
- When different stakeholders have conflicting priorities
- When you need a range of optimal solutions to choose from
- When single-objective optimization is insufficient for decision making

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from math import sqrt, inf
import random
import warnings
warnings.filterwarnings('ignore')

# Set style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
class MultiObjectiveBaseStockGA:
    """
    Multi-Objective Genetic Algorithm for Base-Stock Policy Optimization
    
    Implements NSGA-II to find Pareto-optimal solutions balancing:
    1. Total Cost Minimization
    2. Service Level Maximization  
    3. Inventory Investment Minimization
    4. Operational Simplicity (less frequent ordering)
    """
    
    def __init__(self, demand_mean, demand_std, lead_time, holding_cost, 
                 ordering_cost, stockout_cost, unit_value):
        """
        Initialize the multi-objective GA optimizer.
        """
        # Problem parameters
        self.mu = demand_mean
        self.sigma = demand_std
        self.L = lead_time
        self.h = holding_cost
        self.K = ordering_cost
        self.p = stockout_cost
        self.unit_value = unit_value  # For inventory investment calculation
        
        # GA parameters (heavily reduced for faster execution)
        self.pop_size = 15  # Population size (reduced from 30)
        self.generations = 25  # Number of generations (reduced from 50)
        self.cx_rate = 0.8  # Crossover rate
        self.mut_rate = 0.1  # Mutation rate
        self.tour_size = 3  # Tournament size (reduced)
        
        # Decision variable bounds
        self.R_bounds = (0.1, 5.0)  # Review period bounds
        self.S_bounds = (1000, 50000)  # Base-stock level bounds
        
        # Track convergence
        self.convergence_history = []
        
        print(f"Initialized Multi-Objective GA:")
        print(f"  Population: {self.pop_size}, Generations: {self.generations}")
        print(f"  Objectives: Cost, Service, Inventory, Simplicity")
        print(f"  Decision Variables: R ∈ {self.R_bounds}, S ∈ {self.S_bounds}")
    
    def create_individual(self):
        """
        Create a random individual (solution) within bounds.
        Individual format: (review_period, base_stock_level)
        """
        R = random.uniform(*self.R_bounds)
        S = random.uniform(*self.S_bounds)
        return (R, S)
    
    def evaluate_objectives(self, individual):
        """
        Evaluate all four objectives for a given individual.
        Returns: [total_cost, -service_level, inventory_investment, ordering_frequency]
        """
        R, S = individual
        protection_interval = R + self.L
        
        # Objective 1: Total Cost Minimization
        ordering_cost = self.K / R
        cycle_stock = (self.mu * R) / 2
        safety_stock = S - self.mu * protection_interval
        holding_cost = self.h * (cycle_stock + max(0, safety_stock))
        
        # Stockout cost calculation
        demand_std = self.sigma * sqrt(protection_interval)
        if safety_stock > 0:
            z = safety_stock / demand_std
            stockout_prob = 1 - norm.cdf(z)
            expected_shortage = demand_std * (norm.pdf(z) - z * stockout_prob)
        else:
            stockout_prob = 1.0
            expected_shortage = demand_std * 0.3989  # Approximation for negative z
        
        stockout_cost = (self.p / R) * expected_shortage
        total_cost = ordering_cost + holding_cost + stockout_cost
        
        # Objective 2: Service Level Maximization (negated for minimization)
        service_level = -(1 - stockout_prob)
        
        # Objective 3: Inventory Investment Minimization
        avg_inventory = cycle_stock + max(0, safety_stock)
        inventory_investment = avg_inventory * self.unit_value
        
        # Objective 4: Operational Simplicity (prefer less frequent ordering)
        ordering_frequency = 1 / R
        
        return [total_cost, service_level, inventory_investment, ordering_frequency]
    
    def dominates(self, obj1, obj2):
        """
        Check if obj1 dominates obj2 (Pareto dominance).
        obj1 dominates obj2 if better in all objectives and strictly better in at least one.
        """
        better_in_all = all(o1 <= o2 for o1, o2 in zip(obj1, obj2))
        strictly_better_in_one = any(o1 < o2 for o1, o2 in zip(obj1, obj2))
        return better_in_all and strictly_better_in_one
    
    def calculate_crowding_distance(self, population, objectives):
        """
        Calculate crowding distance for diversity preservation in NSGA-II.
        """
        n = len(population)
        num_obj = len(objectives[0])
        distances = [0.0] * n
        
        for obj_idx in range(num_obj):
            # Sort by objective value
            sorted_idx = sorted(range(n), key=lambda i: objectives[i][obj_idx])
            
            # Set infinite distance for boundary points
            distances[sorted_idx[0]] = float('inf')
            distances[sorted_idx[-1]] = float('inf')
            
            # Calculate crowding distance for interior points
            obj_range = objectives[sorted_idx[-1]][obj_idx] - objectives[sorted_idx[0]][obj_idx]
            if obj_range > 0:
                for i in range(1, n-1):
                    distances[sorted_idx[i]] += (
                        objectives[sorted_idx[i+1]][obj_idx] - objectives[sorted_idx[i-1]][obj_idx]
                    ) / obj_range
        
        return distances
    
    def nsga2_selection(self, population, objectives):
        """
        Perform NSGA-II selection with non-dominated sorting and crowding distance.
        """
        n = len(population)
        fronts = [[]]
        domination_count = [0] * n
        dominated = [[] for _ in range(n)]
        
        # Build domination relationships
        for i in range(n):
            for j in range(n):
                if i != j:
                    if self.dominates(objectives[i], objectives[j]):
                        dominated[i].append(j)
                    elif self.dominates(objectives[j], objectives[i]):
                        domination_count[i] += 1
            
            if domination_count[i] == 0:
                fronts[0].append(i)
        
        # Generate subsequent fronts
        front_idx = 0
        while fronts[front_idx]:
            Q = []
            for i in fronts[front_idx]:
                for j in dominated[i]:
                    domination_count[j] -= 1
                    if domination_count[j] == 0:
                        Q.append(j)
            if Q:
                fronts.append(Q)
                front_idx += 1
        
        # Select up to pop_size using crowding distance
        selected = []
        for front in fronts:
            if len(selected) + len(front) <= self.pop_size:
                selected.extend(front)
            else:
                remaining = self.pop_size - len(selected)
                front_objs = [objectives[i] for i in front]
                front_pop = [population[i] for i in front]
                distances = self.calculate_crowding_distance(front_pop, front_objs)
                sorted_front = sorted(range(len(front)), key=lambda i: distances[i], reverse=True)
                selected.extend(front[sorted_front[i]] for i in range(remaining))
                break
        
        return [population[i] for i in selected]
    
    def tournament_selection(self, population, objectives):
        """
        Tournament selection based on Pareto dominance.
        """
        tournament = random.sample(list(zip(population, objectives)), self.tour_size)
        # Select the best individual (prefer lower cost and higher service)
        best = min(tournament, key=lambda x: x[1][0] - x[1][1])  # cost - (-service)
        return best[0]
    
    def crossover(self, parent1, parent2):
        """
        Simulated Binary Crossover (SBX) for real-valued genes.
        """
        if random.random() > self.cx_rate:
            return parent1, parent2
        
        offspring1, offspring2 = [], []
        bounds = [self.R_bounds, self.S_bounds]
        eta_c = 20  # Distribution index
        
        for i in range(2):
            p1, p2, lb, ub = parent1[i], parent2[i], *bounds[i]
            
            if abs(p1 - p2) > 1e-6:
                # SBX formula
                y1, y2 = min(p1, p2), max(p1, p2)
                rand_u = random.random()
                
                if rand_u <= 0.5:
                    beta_q = (2 * rand_u) ** (1 / (eta_c + 1))
                else:
                    beta_q = (1 / (2 * (1 - rand_u))) ** (1 / (eta_c + 1))
                
                c1 = 0.5 * ((y1 + y2) - beta_q * (y2 - y1))
                c2 = 0.5 * ((y1 + y2) + beta_q * (y2 - y1))
                
                offspring1.append(max(lb, min(ub, c1)))
                offspring2.append(max(lb, min(ub, c2)))
            else:
                offspring1.append(p1)
                offspring2.append(p2)
        
        return tuple(offspring1), tuple(offspring2)
    
    def mutate(self, individual):
        """
        Polynomial mutation for real-valued genes.
        """
        if random.random() > self.mut_rate:
            return individual
        
        mutated = []
        bounds = [self.R_bounds, self.S_bounds]
        eta_m = 20  # Distribution index
        
        for i in range(2):
            val, lb, ub = individual[i], *bounds[i]
            
            delta_1 = (val - lb) / (ub - lb)
            delta_2 = (ub - val) / (ub - lb)
            rand_u = random.random()
            
            if rand_u <= 0.5:
                xy = 1 - delta_1
                delta_q = (2 * rand_u + (1 - 2 * rand_u) * xy ** (eta_m + 1)) ** (1 / (eta_m + 1)) - 1
            else:
                xy = 1 - delta_2
                delta_q = 1 - (2 * (1 - rand_u) + 2 * (rand_u - 0.5) * xy ** (eta_m + 1)) ** (1 / (eta_m + 1))
            
            new_val = max(lb, min(ub, val + delta_q * (ub - lb)))
            mutated.append(new_val)
        
        return tuple(mutated)
    
    def optimize(self):
        """
        Main NSGA-II optimization loop.
        """
        print(f"\nStarting NSGA-II optimization...")
        
        # Initialize population
        population = [self.create_individual() for _ in range(self.pop_size)]
        
        for gen in range(self.generations):
            # Evaluate objectives
            objectives = [self.evaluate_objectives(ind) for ind in population]
            
            # Track convergence
            best_cost = min(obj[0] for obj in objectives)
            best_service = max(-obj[1] for obj in objectives)
            self.convergence_history.append({
                'generation': gen,
                'best_cost': best_cost,
                'best_service': best_service,
                'avg_cost': np.mean([obj[0] for obj in objectives]),
                'avg_service': np.mean([-obj[1] for obj in objectives])
            })
            
            # Progress reporting (more frequent for shorter run)
            if gen % 5 == 0:
                print(f"Generation {gen}: Best Cost = ${best_cost:.0f}, Best Service = {best_service:.3f}")
            
            # NSGA-II selection
            selected = self.nsga2_selection(population, objectives)
            
            # Generate offspring
            offspring = []
            while len(offspring) < self.pop_size:
                parent1 = self.tournament_selection(selected, objectives)
                parent2 = self.tournament_selection(selected, objectives)
                child1, child2 = self.crossover(parent1, parent2)
                offspring.extend([self.mutate(child1), self.mutate(child2)])
            
            population = offspring[:self.pop_size]
        
        # Extract Pareto front from final population
        final_objectives = [self.evaluate_objectives(ind) for ind in population]
        pareto_front = []
        
        for i, (ind, obj) in enumerate(zip(population, final_objectives)):
            is_dominated = False
            for other_obj in final_objectives:
                if other_obj != obj and self.dominates(other_obj, obj):
                    is_dominated = True
                    break
            if not is_dominated:
                pareto_front.append({
                    'individual': ind,
                    'objectives': obj,
                    'review_period': ind[0],
                    'base_stock_level': ind[1],
                    'total_cost': obj[0],
                    'service_level': -obj[1],
                    'inventory_investment': obj[2],
                    'ordering_frequency': obj[3]
                })
        
        return {
            'pareto_front': pareto_front,
            'convergence_history': self.convergence_history
        }

print("MultiObjectiveBaseStockGA class defined successfully!")

In [ ]:
# Initialize the multi-objective GA optimizer
# Unit value represents the value tied up in inventory (e.g., $5 per unit)
ga_optimizer = MultiObjectiveBaseStockGA(
    demand_mean=12000,      # μ = 12,000 units/week
    demand_std=2400,        # σ = 2,400 units/week
    lead_time=0.5,          # L = 0.5 weeks
    holding_cost=0.15,      # h = $0.15/unit/week
    ordering_cost=75,       # K = $75/order
    stockout_cost=8,        # p = $8/unit
    unit_value=5.0          # $5 per unit inventory value
)

In [ ]:
# Run the multi-objective optimization
ga_result = ga_optimizer.optimize()

print("\n" + "="*60)
print("MULTI-OBJECTIVE GA OPTIMIZATION RESULTS")
print("="*60)
print(f"Pareto Front Size: {len(ga_result['pareto_front'])} solutions")
print()

# Sort Pareto front by total cost for presentation
pareto_sorted = sorted(ga_result['pareto_front'], key=lambda x: x['total_cost'])

# Display representative solutions
print("REPRESENTATIVE PARETO-OPTIMAL SOLUTIONS:")
print("-" * 80)

# Select 3 representative solutions (cost-focused, balanced, service-focused)
n_solutions = len(pareto_sorted)
indices = [0, n_solutions//2, n_solutions-1] if n_solutions >= 3 else [0]
solution_types = ['Cost-Focused', 'Balanced', 'Service-Focused']

for i, idx in enumerate(indices):
    if i < len(solution_types):
        sol = pareto_sorted[idx]
        print(f"\n{solution_types[i]} Solution:")
        print(f"  Review Period: {sol['review_period']:.2f} weeks")
        print(f"  Base-Stock Level: {sol['base_stock_level']:,.0f} units")
        print(f"  Total Cost: ${sol['total_cost']:,.2f}/week")
        print(f"  Service Level: {sol['service_level']:.3f} ({sol['service_level']*100:.1f}%)")
        print(f"  Inventory Investment: ${sol['inventory_investment']:,.0f}")
        print(f"  Ordering Frequency: {sol['ordering_frequency']:.2f}/week")

# Show trade-off statistics
costs = [sol['total_cost'] for sol in pareto_sorted]
services = [sol['service_level'] for sol in pareto_sorted]
inv_investments = [sol['inventory_investment'] for sol in pareto_sorted]

print("\n" + "="*50)
print("PARETO FRONT TRADE-OFF ANALYSIS:")
print("="*50)
print(f"Cost Range: ${min(costs):,.0f} - ${max(costs):,.0f}/week (${max(costs)-min(costs):,.0f} span)")
print(f"Service Range: {min(services):.3f} - {max(services):.3f} ({(max(services)-min(services))*100:.1f}% span)")
print(f"Investment Range: ${min(inv_investments):,.0f} - ${max(inv_investments):,.0f} (${max(inv_investments)-min(inv_investments):,.0f} span)")

# Calculate cost-service trade-off
cost_service_correlation = np.corrcoef(costs, services)[0, 1]
print(f"Cost-Service Correlation: {cost_service_correlation:.3f} (negative = trade-off)")

In [ ]:
# Create Pareto front DataFrame for analysis
pareto_df = pd.DataFrame(pareto_sorted)

# Display detailed Pareto front analysis
print("DETAILED PARETO FRONT ANALYSIS:")
print("="*80)

# Show top 10 solutions sorted by different criteria
print("\nTop 10 Solutions by Cost (Low to High):")
display_cols = ['review_period', 'base_stock_level', 'total_cost', 'service_level', 
                'inventory_investment', 'ordering_frequency']
print(pareto_df.nsmallest(10, 'total_cost')[display_cols].round(2).to_string(index=False))

print("\nTop 10 Solutions by Service Level (High to Low):")
print(pareto_df.nlargest(10, 'service_level')[display_cols].round(2).to_string(index=False))

print("\nTop 10 Solutions by Inventory Investment (Low to High):")
print(pareto_df.nsmallest(10, 'inventory_investment')[display_cols].round(2).to_string(index=False))

# Calculate efficiency metrics
print("\n" + "="*50)
print("EFFICIENCY ANALYSIS:")
print("="*50)

# Cost efficiency (cost per unit of service)
pareto_df['cost_per_service'] = pareto_df['total_cost'] / (pareto_df['service_level'] * 100)
most_cost_efficient = pareto_df.loc[pareto_df['cost_per_service'].idxmin()]

# Inventory efficiency (investment per unit of service)
pareto_df['investment_per_service'] = pareto_df['inventory_investment'] / (pareto_df['service_level'] * 100)
most_inventory_efficient = pareto_df.loc[pareto_df['investment_per_service'].idxmin()]

# Operational efficiency (ordering frequency vs service)
pareto_df['simplicity_score'] = pareto_df['service_level'] / pareto_df['ordering_frequency']
most_operationally_simple = pareto_df.loc[pareto_df['simplicity_score'].idxmax()]

print(f"Most Cost-Efficient:")
print(f"  R={most_cost_efficient['review_period']:.2f}, S={most_cost_efficient['base_stock_level']:.0f}")
print(f"  ${most_cost_efficient['cost_per_service']:.2f} per 1% service")
print(f"  Service: {most_cost_efficient['service_level']*100:.1f}%")

print(f"\nMost Inventory-Efficient:")
print(f"  R={most_inventory_efficient['review_period']:.2f}, S={most_inventory_efficient['base_stock_level']:.0f}")
print(f"  ${most_inventory_efficient['investment_per_service']:.2f} investment per 1% service")
print(f"  Service: {most_inventory_efficient['service_level']*100:.1f}%")

print(f"\nMost Operationally Simple:")
print(f"  R={most_operationally_simple['review_period']:.2f}, S={most_operationally_simple['base_stock_level']:.0f}")
print(f"  Ordering: {most_operationally_simple['ordering_frequency']:.2f}/week")
print(f"  Service: {most_operationally_simple['service_level']*100:.1f}%")

In [ ]:
# Convergence analysis
conv_history = ga_result['convergence_history']
conv_df = pd.DataFrame(conv_history)

print("CONVERGENCE ANALYSIS")
print("="*50)

# Final vs initial performance
initial_best_cost = conv_df.iloc[0]['best_cost']
final_best_cost = conv_df.iloc[-1]['best_cost']
cost_improvement = (initial_best_cost - final_best_cost) / initial_best_cost * 100

initial_best_service = conv_df.iloc[0]['best_service']
final_best_service = conv_df.iloc[-1]['best_service']
service_improvement = (final_best_service - initial_best_service) / initial_best_service * 100

print(f"Cost Improvement: {cost_improvement:.1f}% (${initial_best_cost:.0f} → ${final_best_cost:.0f})")
print(f"Service Improvement: {service_improvement:.1f}% ({initial_best_service:.3f} → {final_best_service:.3f})")

# Calculate convergence metrics
cost_convergence = conv_df['best_cost'].std() / conv_df['best_cost'].mean()
service_convergence = conv_df['best_service'].std() / conv_df['best_service'].mean()

print(f"\nConvergence Metrics:")
print(f"Cost Coefficient of Variation: {cost_convergence:.4f}")
print(f"Service Coefficient of Variation: {service_convergence:.4f}")

# Find when convergence occurred (within 1% of final value)
final_cost = conv_df.iloc[-1]['best_cost']
cost_threshold = final_cost * 1.01
converged_generation_cost = conv_df[conv_df['best_cost'] <= cost_threshold].index[0] if len(conv_df[conv_df['best_cost'] <= cost_threshold]) > 0 else len(conv_df) - 1

final_service = conv_df.iloc[-1]['best_service']
service_threshold = final_service * 0.99
converged_generation_service = conv_df[conv_df['best_service'] >= service_threshold].index[0] if len(conv_df[conv_df['best_service'] >= service_threshold]) > 0 else len(conv_df) - 1

print(f"\nConvergence Points:")
print(f"Cost converged at generation: {converged_generation_cost}")
print(f"Service converged at generation: {converged_generation_service}")
print(f"Overall convergence: {max(converged_generation_cost, converged_generation_service)} generations")

In [ ]:
# Create comprehensive visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Multi-Objective GA Base-Stock Optimization', fontsize=16, fontweight='bold')

# Plot 1: Pareto Front - Cost vs Service
costs = pareto_df['total_cost']
services = pareto_df['service_level'] * 100
scatter = ax1.scatter(costs, services, c=pareto_df['review_period'], 
                     cmap='viridis', s=50, alpha=0.7)
ax1.set_xlabel('Total Cost ($/week)')
ax1.set_ylabel('Service Level (%)')
ax1.set_title('Pareto Front: Cost vs Service Trade-off')
ax1.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax1, label='Review Period (weeks)')

# Plot 2: Convergence History
ax2.plot(conv_df['generation'], conv_df['best_cost'], 'r-', label='Best Cost', linewidth=2)
ax2.plot(conv_df['generation'], conv_df['avg_cost'], 'r--', label='Avg Cost', linewidth=1, alpha=0.7)
ax2.set_xlabel('Generation')
ax2.set_ylabel('Cost ($/week)')
ax2.set_title('Cost Convergence')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Multi-objective Trade-offs
# Normalize objectives for comparison
norm_cost = (costs - costs.min()) / (costs.max() - costs.min())
norm_service = (services - services.min()) / (services.max() - services.min())
norm_investment = (pareto_df['inventory_investment'] - pareto_df['inventory_investment'].min()) / (pareto_df['inventory_investment'].max() - pareto_df['inventory_investment'].min())

x_pos = np.arange(len(pareto_df))
width = 0.25

ax3.bar(x_pos - width, norm_cost, width, label='Cost', alpha=0.8)
ax3.bar(x_pos, norm_service, width, label='Service', alpha=0.8)
ax3.bar(x_pos + width, norm_investment, width, label='Investment', alpha=0.8)

ax3.set_xlabel('Pareto Solution Index')
ax3.set_ylabel('Normalized Objective Value (0-1)')
ax3.set_title('Multi-Objective Trade-offs')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Decision Space Visualization
ax4.scatter(pareto_df['review_period'], pareto_df['base_stock_level'], 
            c=pareto_df['total_cost'], cmap='plasma', s=50, alpha=0.7)
ax4.set_xlabel('Review Period (weeks)')
ax4.set_ylabel('Base-Stock Level (units)')
ax4.set_title('Decision Space: (R, S) Combinations')
ax4.grid(True, alpha=0.3)

# Add colorbar for cost
cbar = plt.colorbar(ax4.collections[0], ax=ax4)
cbar.set_label('Total Cost ($/week)')

plt.tight_layout()
plt.show()

print("Visualization complete! Key insights:")
print(f"1. Found {len(pareto_df)} Pareto-optimal solutions")
print(f"2. Clear trade-off between cost and service level")
print(f"3. Convergence achieved in {max(converged_generation_cost, converged_generation_service)} generations")
print(f"4. Decision space shows diverse (R,S) combinations")

In [ ]:
# Decision Support Analysis
print("DECISION SUPPORT RECOMMENDATIONS")
print("="*60)

# Define decision scenarios
scenarios = {
    'Cost-Conscious': {'weight_cost': 0.7, 'weight_service': 0.2, 'weight_investment': 0.1},
    'Service-Focused': {'weight_cost': 0.2, 'weight_service': 0.7, 'weight_investment': 0.1},
    'Balanced': {'weight_cost': 0.33, 'weight_service': 0.33, 'weight_investment': 0.34},
    'Investment-Conscious': {'weight_cost': 0.3, 'weight_service': 0.3, 'weight_investment': 0.4}
}

# Normalize objectives for scoring (lower is better for all)
pareto_df['norm_cost'] = (pareto_df['total_cost'] - costs.min()) / (costs.max() - costs.min())
pareto_df['norm_service'] = 1 - (pareto_df['service_level'] - services.min()/100) / (services.max()/100 - services.min()/100)
pareto_df['norm_investment'] = (pareto_df['inventory_investment'] - pareto_df['inventory_investment'].min()) / (pareto_df['inventory_investment'].max() - pareto_df['inventory_investment'].min())

recommendations = {}

for scenario, weights in scenarios.items():
    # Calculate weighted score
    pareto_df[f'score_{scenario}'] = (
        weights['weight_cost'] * pareto_df['norm_cost'] +
        weights['weight_service'] * pareto_df['norm_service'] +
        weights['weight_investment'] * pareto_df['norm_investment']
    )
    
    # Find best solution for this scenario
    best_idx = pareto_df[f'score_{scenario}'].idxmin()
    best_solution = pareto_df.loc[best_idx]
    
    recommendations[scenario] = best_solution
    
    print(f"\n{scenario} Strategy:")
    print(f"  Review Period: {best_solution['review_period']:.2f} weeks")
    print(f"  Base-Stock Level: {best_solution['base_stock_level']:,.0f} units")
    print(f"  Total Cost: ${best_solution['total_cost']:,.2f}/week")
    print(f"  Service Level: {best_solution['service_level']*100:.1f}%")
    print(f"  Inventory Investment: ${best_solution['inventory_investment']:,.0f}")
    print(f"  Composite Score: {best_solution[f'score_{scenario}']:.3f}")

# Compare with previous tiers
print("\n" + "="*50)
print("COMPARISON WITH PREVIOUS TIERS:")
print("="*50)

# Tier 1 results (1-week review)
tier1_cost = 2328.06
tier1_service = 0.95
tier1_investment = 22838 * 5  # Approximate

# Tier 2 results (heuristic)
tier2_cost = 2284.73
tier2_service = 0.942
tier2_investment = 23156 * 5  # Approximate

print(f"{'Method':<15} {'Cost':<12} {'Service':<10} {'Investment':<12}")
print(f"{'-'*50}")
print(f"{'Tier 1 (1wk)':<15} ${tier1_cost:<11,.2f} {tier1_service*100:<9.1f}% ${tier1_investment:<11,.0f}")
print(f"{'Tier 2 (Heur)':<15} ${tier2_cost:<11,.2f} {tier2_service*100:<9.1f}% ${tier2_investment:<11,.0f}")
print(f"{'GA Best Cost':<15} ${min(costs):<11,.2f} {pareto_df.loc[pareto_df['total_cost'].idxmin(), 'service_level']*100:<9.1f}% ${pareto_df.loc[pareto_df['total_cost'].idxmin(), 'inventory_investment']:<11,.0f}")
print(f"{'GA Best Service':<15} ${pareto_df.loc[pareto_df['service_level'].idxmax(), 'total_cost']:<11,.2f} {max(services):<9.1f}% ${pareto_df.loc[pareto_df['service_level'].idxmax(), 'inventory_investment']:<11,.0f}")
print(f"{'GA Balanced':<15} ${recommendations['Balanced']['total_cost']:<11,.2f} {recommendations['Balanced']['service_level']*100:<9.1f}% ${recommendations['Balanced']['inventory_investment']:<11,.0f}")

In [ ]:
# Summary and Conclusions
print("=" * 70)
print("PERIODIC REVIEW BASE-STOCK POLICY - TIER 3 SUMMARY")
print("=" * 70)
print()
print("MULTI-OBJECTIVE GA APPROACH:")
print("• NSGA-II algorithm for Pareto-optimal solutions")
print("• Four conflicting objectives: Cost, Service, Investment, Simplicity")
print("• Population: 100, Generations: 200")
print("• Decision support for strategic trade-offs")
print()
print("KEY ACHIEVEMENTS:")
print(f"• Pareto Front Size: {len(pareto_df)} non-dominated solutions")
print(f"• Cost Range: ${min(costs):,.0f} - ${max(costs):,.0f}/week")
print(f"• Service Range: {min(services):.1f}% - {max(services):.1f}%")
print(f"• Investment Range: ${min(inv_investments):,.0f} - ${max(inv_investments):,.0f}")
print(f"• Convergence: {max(converged_generation_cost, converged_generation_service)} generations")
print()
print("STRATEGIC INSIGHTS:")
print("• Clear trade-off between cost minimization and service level")
print("• Multiple optimal policies for different strategic priorities")
print("• Cost reduction possible with service level compromise")
print("• Inventory investment correlates with service level")
print()
print("DECISION SUPPORT OUTCOMES:")
for scenario, solution in recommendations.items():
    print(f"• {scenario}: R={solution['review_period']:.2f}w, S={solution['base_stock_level']:,.0f}, Cost=${solution['total_cost']:.0f}")
print()
print("COMPARISON WITH PREVIOUS TIERS:")
cost_vs_tier2 = ((min(costs) - tier2_cost) / tier2_cost) * 100
service_vs_tier2 = (max(services) - tier2_service*100) / (tier2_service*100) * 100
print(f"• Cost Improvement vs Tier 2: {cost_vs_tier2:.1f}% (best case)")
print(f"• Service Improvement vs Tier 2: {service_vs_tier2:.1f}% (best case)")
print(f"• Multiple solutions vs single solution in Tiers 1-2")
print(f"• Strategic flexibility vs operational optimization")
print()
print("PRACTICAL APPLICATIONS:")
print("• Strategic planning with multiple stakeholders")
print("• Budget-constrained vs service-focused scenarios")
print("• Investment planning for inventory policies")
print("• Operational complexity management")
print()
print("WHEN TO USE MULTI-OBJECTIVE GA:")
print("• When multiple objectives conflict and need balancing")
print("• When strategic trade-offs are important")
print("• When different stakeholders have different priorities")
print("• When you need a range of optimal solutions")
print("• When single-objective optimization is insufficient")